# conDitar-dev · CSF1R 소규모 Colab 데모 · v003

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_protein/blob/main/conDitar_CSF1R_Colab.ipynb)

CSF1R 결정구조 8W1L → 사전학습 diffusion으로 새 분자 생성 → 분자 구조·물성 및 Vina 점수 확인 → 논문 CSF1R 억제 실험과 비교 → ZIP 다운로드.

Colab에서 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택합니다. 기본값은 분자 5개, 배치 1개입니다. 저자의 diffusion step과 seed를 유지합니다. T4 실측은 아래 셀에 기록됩니다.

[공식 코드 및 사용 조건](https://github.com/ninglab/conDitar-dev/blob/main/scripts/conDitar/LICENCE.txt) · [논문 Figure 8 및 Table 6](https://arxiv.org/html/2607.12349v1#S5) · [8W1L 결정구조](https://www.rcsb.org/structure/8W1L)

실행 코드는 이 저장소의 `conditar_runtime.py`와 `conditar_workflow.py`에 있습니다. 공식 코드 및 가중치는 실행할 때 각 출처에서 받습니다. 논문 실험값은 논문의 후보 ID에만 연결합니다.

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

demo_dir = Path('/content/tmp_protein_conditar')
if not demo_dir.exists():
    subprocess.run(['git', 'clone', '--branch', 'main', '--single-branch',
                    'https://github.com/HisameOgasahara/tmp_protein.git', str(demo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(demo_dir), 'pull', '--ff-only'], check=True)

sys.path.insert(0, str(demo_dir))
import conditar_runtime as runtime
import conditar_workflow as workflow
importlib.reload(runtime)
importlib.reload(workflow)

context = runtime.install(num_samples=5, batch_size=1)


In [ ]:
workflow.prepare_inputs(context)


### T4에서 3D 후보 생성

포켓 인코더가 8W1L의 결합 부위를 읽고, diffusion 모델이 새 분자의 원자와 3D 좌표를 생성합니다. 생성 시도 중 분자로 복원된 결과가 SDF에 저장됩니다. GPU 사용량은 1초 간격으로 샘플링합니다.

In [ ]:
workflow.generate(context)


In [ ]:
workflow.show_properties(context)


### Vina 계산 점수

공식 평가기가 생성 포즈를 채점하고 국소 최적화합니다. 논문이 사용한 GLIDE와 Sanofi 화합물 검색은 아래 논문 결과에만 포함됩니다. Vina 점수와 실험 IC₅₀는 단위와 의미가 다릅니다.

In [ ]:
workflow.score_vina(context)


### 논문의 CSF1R 기능 억제 실험값

같은 단백질을 다룬 원 논문의 결과입니다. CL-0은 이번 실행의 포켓을 지정하는 결정구조 리간드입니다. CL-1과 CL-2는 생성 모델의 결과를 단서로 화합물 라이브러리에서 검색했고, CL-10과 CL-11은 모델이 직접 생성했습니다. 이번 실행에서 새로 생성한 분자의 실험값은 미측정으로 저장합니다.

In [ ]:
workflow.show_paper_assays(context)


In [ ]:
workflow.download_zip(context)
